In [5]:
import torch
import pandas as pd
import random
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

from sage_encoder import GraphSAGEEncoder
from fusion_model import FusionPredictor
from utils_graph import load_nodes


# ==========================
# Global seed
# ==========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==========================
# Paths
# ==========================

ROOT = Path().resolve().parents[1]

GRAPH_DIR = ROOT / "graphs"
DATA = ROOT / "data" / "data_cleaned"
MODEL_DIR = ROOT / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)


# ==========================
# Load fixed graphs
# ==========================

gcm = torch.load(GRAPH_DIR / "gcm_graph.pt", weights_only=False)
gmd = torch.load(GRAPH_DIR / "gmd_graph.pt", weights_only=False)


# ==========================
# Node maps
# ==========================

circ_map, _ = load_nodes(DATA / "circRNA_nodes_clean.csv", "circRNA")
dis_map, _ = load_nodes(DATA / "disease_nodes_clean.csv", "disease")


# ==========================
# Helper
# ==========================

def predict_edges(edges, circ_fused, dis_fused, predictor):

    circ_idx = edges["circRNA"].map(circ_map).values
    dis_idx = edges["disease"].map(dis_map).values

    circ_batch = circ_fused[circ_idx]
    dis_batch = dis_fused[dis_idx]

    logits = predictor(circ_batch, dis_batch)

    labels = torch.tensor(edges["label"].values, dtype=torch.float)

    return logits, labels


# ==========================
# Store fold results
# ==========================

results = []


# ==========================
# 5-FOLD LOOP
# ==========================

for fold in range(5):

    print(f"\n===== Fold {fold} =====")

    torch.manual_seed(SEED + fold)
    np.random.seed(SEED + fold)
    random.seed(SEED + fold)

    gcd = torch.load(
        GRAPH_DIR / f"gcd_graph_fold{fold}.pt",
        weights_only=False
    )

    train_edges = pd.read_csv(
        DATA / f"circRNA_disease_fold{fold}_train.csv"
    )

    test_edges = pd.read_csv(
        DATA / f"circRNA_disease_fold{fold}_test.csv"
    )

    encoder = GraphSAGEEncoder(
        in_channels=6,
        hidden_channels=64,
        out_channels=64
    )

    predictor = FusionPredictor()

    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(predictor.parameters()),
        lr=0.002,
        weight_decay=1e-4
    )
    criterion = torch.nn.BCEWithLogitsLoss()

    best_auc = 0
    best_aupr = 0
    best_epoch = 0


    # ==========================
    # Epoch loop
    # ==========================

    for epoch in range(50):

        encoder.train()
        predictor.train()

        optimizer.zero_grad()

        # --------------------------
        # Encode graphs
        # --------------------------

        Ecm = encoder(gcm.x, gcm.edge_index)
        Emd = encoder(gmd.x, gmd.edge_index)
        Ecd = encoder(gcd.x, gcd.edge_index)

        # --------------------------
        # Extract views
        # --------------------------

        circ1 = Ecm[:828]
        circ2 = Ecd[:828]

        dis1 = Emd[682:]
        dis2 = Ecd[828:]

        # --------------------------
        # Concat fusion
        # --------------------------

        circ_fused = torch.cat([circ1, circ2], dim=1)
        dis_fused = torch.cat([dis1, dis2], dim=1)

        # --------------------------
        # Train
        # --------------------------

        train_logits, train_labels = predict_edges(
            train_edges,
            circ_fused,
            dis_fused,
            predictor
        )

        loss = criterion(train_logits, train_labels)

        loss.backward()
        optimizer.step()


        # ==========================
        # Evaluate
        # ==========================

        encoder.eval()
        predictor.eval()

        with torch.no_grad():

            Ecm = encoder(gcm.x, gcm.edge_index)
            Emd = encoder(gmd.x, gmd.edge_index)
            Ecd = encoder(gcd.x, gcd.edge_index)

            circ1 = Ecm[:828]
            circ2 = Ecd[:828]

            dis1 = Emd[682:]
            dis2 = Ecd[828:]

            circ_fused = torch.cat([circ1, circ2], dim=1)
            dis_fused = torch.cat([dis1, dis2], dim=1)

            test_logits, test_labels = predict_edges(
                test_edges,
                circ_fused,
                dis_fused,
                predictor
            )

            probs = torch.sigmoid(test_logits).numpy()

            auc = roc_auc_score(test_labels.numpy(), probs)
            aupr = average_precision_score(test_labels.numpy(), probs)

        # --------------------------
        # Save best model
        # --------------------------

        if aupr > best_aupr:

            best_auc = auc
            best_aupr = aupr
            best_epoch = epoch

            torch.save(
                {
                    "encoder": encoder.state_dict(),
                    "predictor": predictor.state_dict()
                },
                MODEL_DIR / f"best_stage2_concat_fold{fold}.pt"
            )

        # --------------------------
        # Logging
        # --------------------------

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch} | "
                f"Loss {loss.item():.4f} | "
                f"AUC {auc:.4f} | "
                f"AUPR {aupr:.4f} | "
            )
        


    # ==========================
    # Fold summary
    # ==========================

    print(
        f"Best Fold {fold} | "
        f"Epoch {best_epoch} | "
        f"AUC {best_auc:.4f} | "
        f"AUPR {best_aupr:.4f}"
    )

    results.append({
        "fold": fold,
        "auc": best_auc,
        "aupr": best_aupr
    })


# ==========================
# Final summary
# ==========================

df = pd.DataFrame(results)

print("\n===== FINAL 5-FOLD RESULTS =====")
print(df)

print("\nMean AUC:", df["auc"].mean())
print("Mean AUPR:", df["aupr"].mean())


===== Fold 0 =====
Epoch 0 | Loss 0.6956 | AUC 0.7612 | AUPR 0.7616 | 
Epoch 10 | Loss 0.5749 | AUC 0.7855 | AUPR 0.7927 | 
Epoch 20 | Loss 0.5507 | AUC 0.8164 | AUPR 0.8083 | 
Epoch 30 | Loss 0.5038 | AUC 0.8073 | AUPR 0.7612 | 
Epoch 40 | Loss 0.4600 | AUC 0.8430 | AUPR 0.7854 | 
Best Fold 0 | Epoch 23 | AUC 0.8240 | AUPR 0.8116

===== Fold 1 =====
Epoch 0 | Loss 0.6944 | AUC 0.8116 | AUPR 0.8068 | 
Epoch 10 | Loss 0.5707 | AUC 0.8012 | AUPR 0.7869 | 
Epoch 20 | Loss 0.5477 | AUC 0.8180 | AUPR 0.8102 | 
Epoch 30 | Loss 0.5020 | AUC 0.8215 | AUPR 0.7739 | 
Epoch 40 | Loss 0.4364 | AUC 0.7870 | AUPR 0.7256 | 
Best Fold 1 | Epoch 1 | AUC 0.8088 | AUPR 0.8155

===== Fold 2 =====
Epoch 0 | Loss 0.6947 | AUC 0.7904 | AUPR 0.7766 | 
Epoch 10 | Loss 0.5753 | AUC 0.7648 | AUPR 0.7304 | 
Epoch 20 | Loss 0.5442 | AUC 0.7951 | AUPR 0.7907 | 
Epoch 30 | Loss 0.5261 | AUC 0.8000 | AUPR 0.7794 | 
Epoch 40 | Loss 0.4612 | AUC 0.8008 | AUPR 0.7490 | 
Best Fold 2 | Epoch 18 | AUC 0.7959 | AUPR 0.7913